In [ ]:
import pandas as pd
from constantes import pasta_data_04_load_inscritos

# Força o Pandas a mostrar todas as linhas para você conseguir rolar e ver todos os estados
pd.set_option('display.max_rows', 300)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("\n" + "="*90)
print("🗺️ MAPA DE DISTRIBUIÇÃO GEOGRÁFICA DO FIES (DESTINO / IES)")
print("="*90)

# --- 1. CARREGAR OS DADOS ---
print("[*] Carregando a base Load de Inscritos...\n")
df = pd.read_parquet(str(pasta_data_04_load_inscritos))

# Verifica se a coluna corrigida existe (senão, usa a original)
coluna_uf = 'uf_ies_corrigida' if 'uf_ies_corrigida' in df.columns else 'uf_ies'

# --- 2. VISÃO MACRO: PIVOT TABLE POR REGIÃO ALVO ---
print("-" * 90)
print("📊 1. VISÃO MACRO: INSCRITOS POR REGIÃO DA IES (Ano e Semestre)")
print("-" * 90)

pivot_regiao = df.pivot_table(
    index=['ano', 'semestre'], 
    columns='regiao_ies_alvo', 
    aggfunc='size', 
    fill_value=0
)

# Adiciona uma coluna de Total para você ver que a matemática bate com os 2.2 milhões
pivot_regiao['TOTAL_BRASIL'] = pivot_regiao.sum(axis=1)
print(pivot_regiao.to_string())


# --- 3. VISÃO MICRO: VOLUMETRIA POR UF ALVO ---
print("\n\n" + "-" * 90)
print(f"📊 2. VISÃO MICRO: INSCRITOS POR UF DE DESTINO ({coluna_uf})")
print("-" * 90)

resumo_uf = (df.groupby(['ano', 'semestre', 'regiao_ies_alvo', coluna_uf])
             .size()
             .reset_index(name='Total_Inscritos'))

# Ordena por Ano, Semestre, e depois do estado com MAIS inscritos para o com MENOS
resumo_uf = resumo_uf.sort_values(by=['ano', 'semestre', 'Total_Inscritos'], ascending=[True, True, False])

print(resumo_uf.to_string(index=False))

print("\n" + "="*90)
print("✅ Agrupamento concluído! Verifique se todos os estados estão com o volume esperado.")
print("="*90 + "\n")


🗺️ MAPA DE DISTRIBUIÇÃO GEOGRÁFICA DO FIES (DESTINO / IES)
[*] Carregando a base Load de Inscritos...

------------------------------------------------------------------------------------------
📊 1. VISÃO MACRO: INSCRITOS POR REGIÃO DA IES (Ano e Semestre)
------------------------------------------------------------------------------------------
regiao_ies_alvo  Centro-Oeste  Nordeste  Norte  Sudeste    Sul  TOTAL_BRASIL
ano  semestre                                                               
2019 1                  54642    234815  80889   269597  56788        696731
     2                  24183     99496  32659    93577  21401        271316
2020 1                  41066    184349  69308   198424  46226        539373
     2                  18031     76158  24619    73898  16796        209502
2021 1                  19007     81655  33160    95805  21391        251018
     2                  18147     79114  28606    84403  19024        229294


---------------------------------